# Phase 1 — Test: lib/ Skeleton + show_source + should_skip

Dieses Notebook prüft, dass:

1. Das `lib/`-Package korrekt importierbar ist
2. `show_source()` aufklappbare Quellcode-Zellen erzeugt
3. `should_skip()` korrekt zwischen animation/statisch, overrides, modes entscheidet

**Voraussetzung:** Dieses Notebook liegt auf derselben Ebene wie der `lib/`-Ordner.
**Falls du es in `notebooks/` oder `kuer/` ausführst,** ändere `_lib_root = Path('.').resolve()` zu `Path('..').resolve()`.


In [ ]:
# ── lib-Import Setup ─────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Auf Projektebene (neben lib/) — anpassen falls Notebook tiefer liegt:
_lib_root = Path('.').resolve()
if str(_lib_root) not in sys.path:
    sys.path.insert(0, str(_lib_root))

# Optional: Autoreload aktivieren (praktisch während der lib-Entwicklung)
%load_ext autoreload
%autoreload 2

print(f'sys.path enthält: {_lib_root}')
print(f'lib/ existiert:   {(_lib_root / "lib").exists()}')


## Test 1 — Imports funktionieren


In [ ]:
from lib.plotting import show_source, should_skip
from lib import __version__ as lib_version

print(f'✅ lib/ Version {lib_version} geladen')
print(f'   show_source:  {show_source}')
print(f'   should_skip:  {should_skip}')


## Test 2 — `show_source()` zeigt Quellcode aufklappbar

Unter dieser Zelle sollte ein **zugeklapptes** `<details>`-Element erscheinen:  
🔎 *Quellcode: `should_skip` (aus `lib/plotting.py`)*  

Beim Klick darauf öffnet sich der Quellcode syntax-highlighted.

In [ ]:
show_source(should_skip)


## Test 3 — `show_source()` mit eigenem Titel


In [ ]:
show_source(show_source, title='Das ist der Helper selbst — rekursive Transparenz ;)')


## Test 4 — `should_skip()` Entscheidungs-Matrix

Wir prüfen alle relevanten Pfade:

| Fall | out_path | asset_type | name | overrides | erwartetes Ergebnis |
|---|---|---|---|---|---|
| 1 | existiert | animation | — | — | `True` (skip) |
| 2 | fehlt | animation | — | — | `False` (rendern) |
| 3 | existiert | statisch | — | — | `False` (default always) |
| 4 | existiert | statisch | matched override | skip | `True` (skip) |
| 5 | existiert | animation | override force | force | `False` (rendern) |


In [ ]:
import os, tempfile

# Test-Dateien
_tmp = tempfile.mkdtemp()
_exists = os.path.join(_tmp, 'exists.gif')
_missing = os.path.join(_tmp, 'missing.gif')
open(_exists, 'w').write('dummy')

# Test-Config
CFG = {
    'animation': {
        'modus':          'skip_if_exists',
        'modus_statisch': 'always',
        'overrides': {
            'my_expensive_map': 'skip_if_exists',
            'my_dev_chart':     'force_rebuild',
        }
    }
}

tests = [
    (_exists,  'animation', 'some_anim',         True,  'Fall 1: anim exists, default skip'),
    (_missing, 'animation', 'some_anim',         False, 'Fall 2: anim fehlt'),
    (_exists,  'statisch',  'some_chart',        False, 'Fall 3: stat exists, default always'),
    (_exists,  'statisch',  'my_expensive_map',  True,  'Fall 4: stat exists, override skip'),
    (_exists,  'animation', 'my_dev_chart',      False, 'Fall 5: anim exists, override force'),
]

for out, typ, name, expected, desc in tests:
    got = should_skip(out, typ, name, CFG)
    status = '✅' if got == expected else '❌'
    print(f'{status} {desc:<40s} got={got} expected={expected}')


## Test 5 — Master-Schalter `skip_if_exists_all`

Setzt man `CFG['animation']['modus_statisch'] = 'skip_if_exists_all'`, werden
**alle** statischen Charts auf `skip_if_exists` umgeschaltet (ohne overrides
einzeln setzen zu müssen).

In [ ]:
CFG_master = {
    'animation': {
        'modus': 'skip_if_exists',
        'modus_statisch': 'skip_if_exists_all',
    }
}

got = should_skip(_exists, 'statisch', 'any_chart_name', CFG_master)
print(f'{"✅" if got else "❌"} Master-Schalter skip_if_exists_all → {got}')


## Test 6 — Robustheit gegen fehlerhafte Inputs


In [ ]:
# Falsch getippter Modus → Warning + Fallback
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    got = should_skip(_exists, 'animation', 'x', {'animation': {'modus': 'typo'}})
    print(f'{"✅" if got == False and len(w) == 1 else "❌"} Typo im Modus → {len(w)} Warning(s), Ergebnis {got}')

# asset_type falsch → ValueError
try:
    should_skip(_exists, 'video', 'x', CFG)
    print('❌ Hätte ValueError werfen sollen')
except ValueError as e:
    print(f'✅ ValueError bei asset_type=video: {e}')


## Abschluss

Wenn alle Tests oben ✅ zeigen und die `show_source`-Zellen die Quellcodes
aufklappbar anzeigen, ist **Phase 1** erfolgreich abgeschlossen.

**Nächster Schritt:** Phase 2 (Anker-Hygiene) — mechanische Bereinigung von
ID-Ankern mit führenden Ziffern in 8 Notebooks.
